# nb_mta_refresh_copy

Runs after the Copy activity has overwritten `bronze.mta_hourly_2025`.
Order: Silver (2025 onward), Gold dimensions (stable keys), Gold fact (2025 onward), integrity checks.
The last cell raises an error if any check fails, so the pipeline stops before the semantic model refresh.
Attach `mta_subway_lakehouse` and set it as the default Lakehouse before running.

## 1. Silver: replace 2025 onward

In [13]:
import io, requests, pandas as pd
URL = "https://data.ny.gov/resource/5wq4-mkjj"
r = requests.get(URL + ".csv", params={"$limit": 3, "$order": "transit_timestamp DESC"}, timeout=120)
print("status:", r.status_code)
sample = pd.read_csv(io.StringIO(r.text), dtype=str, keep_default_na=False)
print(list(sample.columns))
print(sample.head(3).T)
mx = requests.get(URL + ".json", params={"$select": "max(transit_timestamp)"}, timeout=120).json()
print("latest timestamp at the source:", mx)

StatementMeta(, 842e9817-ddb8-4fcc-9c22-cbfb7028d15e, 16, Finished, Available, Finished, False)

status: 200
['transit_timestamp', 'transit_mode', 'station_complex_id', 'station_complex', 'borough', 'payment_method', 'fare_class_category', 'ridership', 'transfers', 'latitude', 'longitude', 'georeference']
                                                0  \
transit_timestamp         2026-10-08T23:00:00.000   
transit_mode                               subway   
station_complex_id                            418   
station_complex                      233 St (2,5)   
borough                                     Bronx   
payment_method                               omny   
fare_class_category                  OMNY - Other   
ridership                                     2.0   
transfers                                     0.0   
latitude                                40.893192   
longitude                              -73.857475   
georeference         POINT (-73.857475 40.893192)   

                                               1                            2  
transit_timestamp   

## 2. Gold dimensions: keys stay fixed once assigned

In [14]:
import io, time, datetime as dt
import requests
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

DRY_RUN = False            # True: pull 2 pages, show a sample, write nothing. Set False for the real run.
DATASET = "5wq4-mkjj"
URL = f"https://data.ny.gov/resource/{DATASET}"
PAGE = 50000
FLUSH_EVERY = 10          # pages per write to the staging table (about 500,000 rows)
APP_TOKEN = "9P7pkWDJND0Q6dfywRhhzzfzg"            # paste your Socrata app token
HEADERS = {"X-App-Token": APP_TOKEN} if APP_TOKEN else {}
ORDER = "transit_timestamp,station_complex_id,payment_method,fare_class_category"
TS_FMT = "MM/dd/yyyy hh:mm:ss a"
BRONZE = "bronze.mta_hourly_2025"
STAGE = "bronze.mta_hourly_2025_stage"

last_ts = spark.table(BRONZE).agg(F.max(F.to_timestamp("transit_timestamp", TS_FMT))).first()[0]
assert last_ts is not None, "Bronze is empty; restore it before loading"
where = f"transit_timestamp > '{last_ts.strftime('%Y-%m-%dT%H:%M:%S')}'"
print("last loaded timestamp:", last_ts)

def get(ext, params):
    last = None
    for attempt in range(6):
        try:
            r = requests.get(f"{URL}.{ext}", params=params, headers=HEADERS, timeout=300)
            if r.status_code == 200:
                return r
            last = f"HTTP {r.status_code}"
        except requests.RequestException as e:
            last = str(e)
        time.sleep(15 * (attempt + 1))
    raise RuntimeError(f"request failed after retries: {last}")

expected = int(get("json", {"$select": "count(*)", "$where": where}).json()[0]["count"])
print("new rows reported by the source:", expected)

need = ["transit_timestamp", "transit_mode", "station_complex_id", "station_complex", "borough",
        "payment_method", "fare_class_category", "ridership", "transfers", "latitude", "longitude"]
out_cols = need + ["Georeference", "source_dataset", "load_date"]
schema = StructType([StructField(c, StringType(), True) for c in out_cols])
load_date = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def prepare(frames):
    pdf = pd.concat(frames, ignore_index=True)
    pdf.columns = [c.strip().lower() for c in pdf.columns]
    missing = [c for c in need if c not in pdf.columns]
    assert not missing, f"columns missing from the API response: {missing}"
    pdf["Georeference"] = pdf["georeference"] if "georeference" in pdf.columns else None
    pdf["transit_timestamp"] = pd.to_datetime(pdf["transit_timestamp"]).dt.strftime("%m/%d/%Y %I:%M:%S %p")
    pdf["source_dataset"] = DATASET
    pdf["load_date"] = load_date
    return spark.createDataFrame(pdf[out_cols], schema=schema)

if expected == 0:
    print("Nothing new at the source; Bronze is up to date.")
else:
    first_flush, frames, pages, offset = True, [], 0, 0
    while True:
        txt = get("csv", {"$where": where, "$order": ORDER, "$limit": PAGE, "$offset": offset}).text
        if not txt.strip():
            break
        page = pd.read_csv(io.StringIO(txt), dtype=str, keep_default_na=False)
        if page.empty:
            break
        frames.append(page)
        pages += 1
        offset += PAGE
        if DRY_RUN and pages >= 2:
            break
        if pages % FLUSH_EVERY == 0:
            prepare(frames).write.format("delta").mode("overwrite" if first_flush else "append").saveAsTable(STAGE)
            first_flush, frames = False, []
        if len(page) < PAGE:
            break

    if DRY_RUN:
        prepare(frames).show(5, truncate=False)
        print("DRY RUN: nothing was written. Set DRY_RUN = False for the real load.")
    else:
        if frames:
            prepare(frames).write.format("delta").mode("overwrite" if first_flush else "append").saveAsTable(STAGE)
        stage = spark.table(STAGE)
        staged = stage.count()
        added = stage.agg(F.sum(F.col("ridership").cast("double"))).first()[0]
        assert staged == expected, f"staged {staged} rows but the source reports {expected}; nothing was appended"
        stage.write.format("delta").mode("append").saveAsTable(BRONZE)
        print("appended to Bronze:", staged, "rows | ridership added:", int(added))

StatementMeta(, 842e9817-ddb8-4fcc-9c22-cbfb7028d15e, 17, Finished, Available, Finished, False)

last loaded timestamp: 2026-10-08 23:00:00
new rows reported by the source: 0
Nothing new at the source; Bronze is up to date.


## 3. Gold fact: replace 2025 onward

In [15]:
from pyspark.sql import functions as F

TS_FMT = "MM/dd/yyyy hh:mm:ss a"
BRONZE = "bronze.mta_hourly_2025"

bad_ts = spark.table(BRONZE).filter(F.to_timestamp("transit_timestamp", TS_FMT).isNull()).count()
assert bad_ts == 0, f"{bad_ts} Bronze rows have an unparsed timestamp"

silver_max = spark.table("silver.ridership_hourly").agg(F.max("transit_ts")).first()[0]
new_rows = (spark.table(BRONZE)
    .withColumn("transit_ts", F.to_timestamp("transit_timestamp", TS_FMT))
    .filter(F.col("transit_ts") > F.lit(silver_max)))

silver_new = (new_rows
    .withColumn("ridership_date", F.to_date("transit_ts"))
    .withColumn("ridership_year", F.year("transit_ts"))
    .withColumn("hour_of_day", F.hour("transit_ts"))
    .withColumn("transit_mode",
        F.when(F.col("transit_mode") == "subway", "Subway")
         .when(F.col("transit_mode") == "staten_island_railway", "Staten Island Railway")
         .when(F.col("transit_mode") == "tram", "Roosevelt Island Tram")
         .otherwise(F.col("transit_mode")))
    .withColumn("payment_method",
        F.when(F.col("payment_method") == "omny", "OMNY")
         .when(F.col("payment_method") == "metrocard", "MetroCard")
         .otherwise(F.col("payment_method")))
    .withColumn("fare_class_category",
        F.regexp_replace("fare_class_category", "^Metrocard", "MetroCard"))
    .withColumn("ridership", F.col("ridership").cast("double").cast("long"))
    .withColumn("transfers", F.col("transfers").cast("double").cast("long"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("load_date", F.to_timestamp(F.substring("load_date", 1, 19)))
    .select("transit_ts", "ridership_date", "ridership_year", "hour_of_day",
            "transit_mode", "station_complex_id", "station_complex", "borough",
            "payment_method", "fare_class_category", "ridership", "transfers",
            "latitude", "longitude", "source_dataset", "load_date"))

n = silver_new.count()
if n == 0:
    print("Silver is already up to date")
else:
    silver_new.write.format("delta").mode("append").saveAsTable("silver.ridership_hourly")
    print("Silver rows appended:", n)

StatementMeta(, 842e9817-ddb8-4fcc-9c22-cbfb7028d15e, 18, Finished, Available, Finished, False)

Silver is already up to date


## 4. Integrity checks (raises an error if any check fails)

In [16]:
import datetime as dt
from pyspark.sql import functions as F

s = spark.table("silver.ridership_hourly")
st = spark.table("gold.dim_station").select("station_key", "station_complex_id")
fc = spark.table("gold.dim_fare_class").select("fare_class_key", "fare_class_category")

gold_last_key = spark.table("gold.fact_ridership_daily").agg(F.max("date_key")).first()[0]
start_date = dt.datetime.strptime(str(gold_last_key), "%Y%m%d").date()
recent = s.filter(F.col("ridership_date") >= F.lit(start_date))

daily = (recent.groupBy("ridership_date", "station_complex_id", "fare_class_category")
               .agg(F.sum("ridership").alias("ridership"), F.sum("transfers").alias("transfers")))
hours = (recent.groupBy("ridership_date", "station_complex_id")
               .agg(F.countDistinct("hour_of_day").cast("int").alias("hours_with_data")))

fact_new = (daily.join(hours, ["ridership_date", "station_complex_id"])
    .join(st, "station_complex_id").join(fc, "fare_class_category")
    .withColumn("date_key", F.date_format("ridership_date", "yyyyMMdd").cast("int"))
    .select("date_key", "station_key", "fare_class_key", "ridership", "transfers", "hours_with_data"))

(fact_new.write.format("delta").mode("overwrite")
    .option("replaceWhere", f"date_key >= {gold_last_key}")
    .saveAsTable("gold.fact_ridership_daily"))
print("Gold fact replaced from", start_date)

StatementMeta(, 842e9817-ddb8-4fcc-9c22-cbfb7028d15e, 19, Finished, Available, Finished, False)

Gold fact replaced from 2026-10-08


In [17]:
from pyspark.sql import functions as F, Window

s = spark.table("silver.ridership_hourly")

# ---------- dim_station, stable keys ----------
variants = (s.groupBy("station_complex_id", "source_dataset", "station_complex",
                      "borough", "transit_mode", "latitude", "longitude")
              .agg(F.count("*").alias("n"),
                   F.min("transit_ts").alias("first_seen"),
                   F.max("transit_ts").alias("last_seen")))
w_ds = Window.partitionBy("station_complex_id", "source_dataset")
w_pick = Window.partitionBy("station_complex_id") \
               .orderBy(F.col("ds_last").desc(), F.col("n").desc())
picked = (variants.withColumn("ds_last", F.max("last_seen").over(w_ds))
                  .withColumn("rn", F.row_number().over(w_pick)).filter("rn = 1")
                  .select("station_complex_id", "station_complex", "borough",
                          "transit_mode", "latitude", "longitude"))
span = (variants.groupBy("station_complex_id")
                .agg(F.min("first_seen").alias("first_seen"),
                     F.max("last_seen").alias("last_seen")))
cur = (picked.join(span, "station_complex_id")
         .withColumn("routes", F.regexp_extract("station_complex", r"\(([^)]*)\)\s*$", 1)))

old_keys = spark.table("gold.dim_station").select("station_complex_id", "station_key")
max_key = old_keys.agg(F.max("station_key")).first()[0] or 0
new_keys = (cur.select("station_complex_id")
              .join(old_keys, "station_complex_id", "left_anti")
              .withColumn("station_key",
                  (F.row_number().over(Window.orderBy("station_complex_id")) + F.lit(max_key)).cast("int")))
keys = old_keys.unionByName(new_keys).cache()
print("new stations:", new_keys.count(), "| total:", keys.count())

dim_station = (cur.join(keys, "station_complex_id")
    .select("station_key", "station_complex_id", "station_complex", "routes",
            "borough", "transit_mode", "latitude", "longitude", "first_seen", "last_seen"))
dim_station.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_station")

# ---------- dim_fare_class, stable keys ----------
fc_cur = (s.groupBy("fare_class_category", "payment_method")
    .agg(F.min("ridership_date").alias("first_seen"), F.max("ridership_date").alias("last_seen"))
    .withColumn("fare_group",
        F.when(F.col("fare_class_category").contains("Unlimited"), "Unlimited")
         .when(F.col("fare_class_category").contains("Seniors"), "Seniors & Disability")
         .when(F.col("fare_class_category").contains("Students"), "Students")
         .when(F.col("fare_class_category").contains("Fair Fare"), "Fair Fare")
         .when(F.col("fare_class_category").contains("Full Fare"), "Full Fare")
         .otherwise("Other")))
old_fc = spark.table("gold.dim_fare_class").select("fare_class_category", "fare_class_key")
max_fc = old_fc.agg(F.max("fare_class_key")).first()[0] or 0
new_fc = (fc_cur.select("fare_class_category")
            .join(old_fc, "fare_class_category", "left_anti")
            .withColumn("fare_class_key",
                (F.row_number().over(Window.orderBy("fare_class_category")) + F.lit(max_fc)).cast("int")))
fc_keys = old_fc.unionByName(new_fc).cache()
print("new fare classes:", new_fc.count(), "| total:", fc_keys.count())

dim_fare = (fc_cur.join(fc_keys, "fare_class_category")
    .select("fare_class_key", "fare_class_category", "payment_method",
            "fare_group", "first_seen", "last_seen"))
dim_fare.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_fare_class")

StatementMeta(, 842e9817-ddb8-4fcc-9c22-cbfb7028d15e, 20, Finished, Available, Finished, False)

new stations: 0 | total: 428
new fare classes: 0 | total: 12


In [18]:
import datetime as dt
import requests
from pyspark.sql import functions as F, Row

TS_FMT = "MM/dd/yyyy hh:mm:ss a"
fact   = spark.table("gold.fact_ridership_daily")
silver = spark.table("silver.ridership_hourly")
bz     = spark.table("bronze.mta_hourly_2025")
sv     = silver.filter("source_dataset = '5wq4-mkjj'")

results = []
def check(name, passed, detail=""):
    results.append((name, "PASS" if passed else "FAIL", str(detail)))
def info(name, detail=""):
    results.append((name, "INFO", str(detail)))
def dup_count(df, cols):
    return df.groupBy(*cols).count().filter("count > 1").count()

bz_n, sv_n = bz.count(), sv.count()
check("Bronze vs Silver: same row count for the 2025 dataset", bz_n == sv_n, f"{bz_n} vs {sv_n}")

dup_bz = dup_count(bz, ["transit_timestamp", "station_complex_id", "payment_method", "fare_class_category"])
check("Bronze: no duplicate keys", dup_bz == 0, f"duplicate keys: {dup_bz}")

dup_sv = dup_count(sv, ["transit_ts", "station_complex_id", "payment_method", "fare_class_category"])
check("Silver: no duplicate keys (2025 dataset)", dup_sv == 0, f"duplicate keys: {dup_sv}")

bz_last = bz.agg(F.max(F.to_timestamp("transit_timestamp", TS_FMT))).first()[0]
sv_last = sv.agg(F.max("transit_ts")).first()[0]
check("Silver is up to date with Bronze", bz_last == sv_last, f"{sv_last} vs {bz_last}")

gold_last = fact.agg(F.max("date_key")).first()[0]
sv_last_key = sv.agg(F.max(F.date_format("ridership_date", "yyyyMMdd").cast("int"))).first()[0]
check("Gold is up to date with Silver", gold_last == sv_last_key, f"{gold_last} vs {sv_last_key}")

src_n = int(requests.get("https://data.ny.gov/resource/5wq4-mkjj.json",
            params={"$select": "count(*)", "$where": f"transit_timestamp <= '{bz_last.strftime('%Y-%m-%dT%H:%M:%S')}'"},
            timeout=300).json()[0]["count"])
info("drift: source rows up to the last loaded hour minus Bronze rows (not zero means revisions)", src_n - bz_n)

run_ts = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
res = spark.createDataFrame([Row(run_ts=run_ts, check_name=n, status=s, detail=d) for n, s, d in results])
res.write.format("delta").mode("append").saveAsTable("silver.dq_results")
res.select("status", "check_name", "detail").show(20, truncate=False)
failed = res.filter("status = 'FAIL'").count()
print("FAILED:", failed)
if failed:
    raise Exception(f"{failed} integrity checks failed")

StatementMeta(, 842e9817-ddb8-4fcc-9c22-cbfb7028d15e, 21, Finished, Available, Finished, False)

+------+------------------------------------------------------------------------------------------+------------------------------------------+
|status|check_name                                                                                |detail                                    |
+------+------------------------------------------------------------------------------------------+------------------------------------------+
|PASS  |Bronze vs Silver: same row count for the 2025 dataset                                     |46363779 vs 46363779                      |
|PASS  |Bronze: no duplicate keys                                                                 |duplicate keys: 0                         |
|PASS  |Silver: no duplicate keys (2025 dataset)                                                  |duplicate keys: 0                         |
|PASS  |Silver is up to date with Bronze                                                          |2026-10-08 23:00:00 vs 2026-10-08 23:00:00|

In [19]:
12155332038 + 32603293

StatementMeta(, 842e9817-ddb8-4fcc-9c22-cbfb7028d15e, 22, Finished, Available, Finished, False)

12187935331